# Week 13: Mechanical Waves & Sound — PHASE 5: Oscillations & Waves

*📚 Physics I (PHY101) · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

---
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Describe** the mathematical form of a travelling wave and identify its parameters (amplitude, wavelength, frequency, wave speed)
2. **Derive** the wave speed on a string from tension and linear mass density
3. **Explain** superposition, constructive and destructive interference
4. **Visualise** two-source interference patterns in 2-D
5. **Analyse** standing waves and resonance on a string
6. **Model** sound as a longitudinal pressure wave and relate intensity to decibels
7. **Apply** wave concepts to vibration and acoustics problems relevant to engineering

## 🎯 Core Mastery Connection

Waves carry energy through space without transporting matter. This week you predict wave speed from medium properties, determine interference patterns from path differences, and calculate standing-wave frequencies from boundary conditions. The core workflow still applies: diagram the wave setup, identify the governing principle (superposition, boundary conditions), write the wave equation, and predict speed, interference, and standing waves.

---
## 🧭 Three-Hour Interactive Studio Plan

**Audience:** Mechatronics Engineering and Computer Engineering students  
**Weekly focus:** Week 13: Mechanical Waves & Sound — PHASE 5: Oscillations & Waves

- **Mechatronics lens:** vibration monitoring, acoustics, and ultrasonic sensing.
- **Computer Engineering lens:** audio, sampling, communications, and wave computation.

| Time | Learning cycle |
|---|---|
| 00:00–00:10 | Launch question, prior-knowledge retrieval, outcomes |
| 00:10–00:50 | Concept cycle 1: explain → predict → test |
| 00:50–01:00 | Checkpoint 1, student questions, peer explanation |
| 01:00–01:10 | Break |
| 01:10–01:50 | Concept cycle 2: worked example → variation → discussion |
| 01:50–02:00 | Checkpoint 2 and misconception repair |
| 02:00–02:10 | Break |
| 02:10–02:40 | Core in-class practice with instructor circulation |
| 02:40–02:50 | Checkpoint 3: exam bridge and professional transfer |
| 02:50–03:00 | Open questions, summary, and exit ticket |

The official start and finish times are followed as published in the timetable. Ask questions at any point; the scheduled checkpoints guarantee additional question time. Checkpoints are private self-checks in this runtime—no identity, upload, homework, or instructor dashboard.


In [ ]:
# Run once. This pulse stays only in the current Colab runtime.
_studio_pulses = {}

def studio_pulse(number, response, minimum_words=8):
    words = str(response).strip().split()
    ready = len(words) >= minimum_words
    _studio_pulses[int(number)] = ready
    if ready:
        print(f"✅ Checkpoint {number}: explanation recorded locally ({len(words)} words).")
    else:
        print(f"🟡 Checkpoint {number}: explain your reasoning in at least {minimum_words} words, then retry.")
    print("Nothing is transmitted or stored for grading.")
    return ready

print("✅ Local studio checkpoints ready")

---
## 📦 Setup

Run this cell first to import all libraries we need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import cm
from IPython.display import HTML, display, Markdown
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, HBox, VBox

%matplotlib inline

# For Colab compatibility
try:
    import google.colab
    IN_COLAB = True
    from matplotlib import rc
    rc('animation', html='jshtml')
except ImportError:
    IN_COLAB = False

plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 5)})
print("\u2705 All libraries loaded successfully!")

---
## 1. What is a Wave?

A **wave** is a disturbance that transfers **energy** without transferring **matter**.

| Property | Symbol | Unit | Meaning |
|----------|--------|------|---------|
| Amplitude | $A$ | m | Maximum displacement from equilibrium |
| Wavelength | $\lambda$ | m | Distance between two consecutive identical points |
| Frequency | $f$ | Hz | Number of complete cycles per second |
| Period | $T = 1/f$ | s | Time for one complete cycle |
| Wave speed | $v = \lambda f$ | m/s | Speed at which the disturbance travels |
| Wave number | $k = 2\pi/\lambda$ | rad/m | Spatial frequency |
| Angular frequency | $\omega = 2\pi f$ | rad/s | Temporal frequency |

### The Travelling Wave Equation

$$y(x,t) = A \sin(kx - \omega t + \phi)$$

- $kx - \omega t$ : wave travelling in the **+x** direction
- $kx + \omega t$ : wave travelling in the **-x** direction
- $\phi$ : initial phase

### Wave Speed on a String

$$v = \sqrt{\frac{T}{\mu}}$$

where $T$ is the tension (N) and $\mu$ is the linear mass density (kg/m).

> **Engineering Analogy:** Think of a wave on a string like a ripple of information travelling through a power line. The tighter the cable (higher tension), the faster disturbances travel.

---
## 🎬 Interactive Demo 1: Animated Transverse Wave

Watch how a sinusoidal wave propagates along a string. Use the sliders to change wavelength ($\lambda$) and frequency ($f$) and observe how the wave speed $v = \lambda f$ changes.

In [ ]:
def transverse_wave_animation(wavelength=2.0, frequency=1.0, amplitude=1.0):
    """Animate a transverse wave with given parameters."""
    fig, ax = plt.subplots(figsize=(11, 4))
    x = np.linspace(0, 10, 500)
    k = 2 * np.pi / wavelength
    omega = 2 * np.pi * frequency
    v = wavelength * frequency

    line, = ax.plot([], [], 'b-', lw=2.5)
    dot, = ax.plot([], [], 'ro', ms=10, zorder=5)
    time_text = ax.text(0.02, 0.92, '', transform=ax.transAxes, fontsize=11,
                        bbox=dict(boxstyle='round', facecolor='lightyellow'))

    ax.set_xlim(0, 10)
    ax.set_ylim(-1.8, 1.8)
    ax.set_xlabel('Position x (m)')
    ax.set_ylabel('Displacement y (m)')
    ax.set_title(f'Transverse Wave:  $\\lambda$={wavelength:.1f} m,  f={frequency:.1f} Hz,  v={v:.1f} m/s')
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    ax.grid(True, alpha=0.3)

    # Draw wavelength annotation
    ax.annotate('', xy=(wavelength, 1.5), xytext=(0, 1.5),
                arrowprops=dict(arrowstyle='<->', color='green', lw=2))
    ax.text(wavelength/2, 1.6, f'$\\lambda$ = {wavelength:.1f} m', ha='center',
            color='green', fontsize=11, fontweight='bold')

    n_frames = 80
    T_total = 2.0 / frequency if frequency > 0 else 2.0

    def init():
        line.set_data([], [])
        dot.set_data([], [])
        time_text.set_text('')
        return line, dot, time_text

    def animate(i):
        t = i * T_total / n_frames
        y = amplitude * np.sin(k * x - omega * t)
        line.set_data(x, y)
        # Track a single point on the string
        x_pt = 5.0
        y_pt = amplitude * np.sin(k * x_pt - omega * t)
        dot.set_data([x_pt], [y_pt])
        time_text.set_text(f't = {t:.2f} s')
        return line, dot, time_text

    ani = animation.FuncAnimation(fig, animate, init_func=init,
                                   frames=n_frames, interval=40, blit=True)
    plt.close(fig)
    return ani

# Interactive widget version
@interact(wavelength=FloatSlider(min=0.5, max=5.0, step=0.25, value=2.0, description='$\\lambda$ (m)'),
          frequency=FloatSlider(min=0.25, max=3.0, step=0.25, value=1.0, description='f (Hz)'),
          amplitude=FloatSlider(min=0.2, max=1.5, step=0.1, value=1.0, description='A (m)'))
def show_wave(wavelength, frequency, amplitude):
    ani = transverse_wave_animation(wavelength, frequency, amplitude)
    display(HTML(ani.to_jshtml()))

---
## 2. Superposition Principle

When two or more waves overlap in the same medium, the **net displacement** is the **algebraic sum** of the individual displacements:

$$y_{\text{net}}(x,t) = y_1(x,t) + y_2(x,t)$$

| Type | Condition | Result |
|------|-----------|--------|
| **Constructive** interference | Waves in phase ($\Delta\phi = 0, 2\pi, \dots$) | Amplitude doubles |
| **Destructive** interference | Waves out of phase ($\Delta\phi = \pi, 3\pi, \dots$) | Amplitude cancels |
| **Partial** interference | Other phase differences | Intermediate amplitude |

---
## 🎬 Interactive Demo 2: Superposition of Two Waves

See how two waves combine. Adjust the frequency and phase of the second wave to observe constructive and destructive interference.

In [ ]:
def superposition_animation(f2=1.0, phase_shift=0.0):
    """Animate the superposition of two waves."""
    fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
    x = np.linspace(0, 10, 500)
    f1 = 1.0
    lam = 2.0
    k = 2 * np.pi / lam
    omega1 = 2 * np.pi * f1
    omega2 = 2 * np.pi * f2

    line1, = axes[0].plot([], [], 'b-', lw=2, label='Wave 1')
    line2, = axes[1].plot([], [], 'r-', lw=2, label='Wave 2')
    line3, = axes[2].plot([], [], 'purple', lw=2.5, label='Superposition')

    for i, (ax, title) in enumerate(zip(axes, ['Wave 1 (reference)', f'Wave 2 (f={f2:.1f} Hz, $\\Delta\\phi$={phase_shift:.1f} rad)', 'Superposition = Wave 1 + Wave 2'])):
        ax.set_xlim(0, 10)
        ax.set_ylim(-2.5, 2.5)
        ax.axhline(0, color='gray', ls='--', lw=0.5)
        ax.set_title(title, fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right')
    axes[2].set_xlabel('Position x (m)')
    fig.tight_layout()

    n_frames = 80
    T_total = 2.0

    def animate(i):
        t = i * T_total / n_frames
        y1 = np.sin(k * x - omega1 * t)
        y2 = np.sin(k * x - omega2 * t + phase_shift)
        line1.set_data(x, y1)
        line2.set_data(x, y2)
        line3.set_data(x, y1 + y2)
        return line1, line2, line3

    ani = animation.FuncAnimation(fig, animate, frames=n_frames, interval=40, blit=True)
    plt.close(fig)
    return ani

@interact(f2=FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0, description='f$_2$ (Hz)'),
          phase_shift=FloatSlider(min=0, max=2*np.pi, step=0.1, value=0, description='$\\Delta\\phi$ (rad)'))
def show_superposition(f2, phase_shift):
    ani = superposition_animation(f2, phase_shift)
    display(HTML(ani.to_jshtml()))

---
### ⏱️ Checkpoint 1 of 3 — Think · Pair · Explain

For **Week 13: Mechanical Waves & Sound — PHASE 5: Oscillations & Waves**, name the governing principle and define its symbols with SI units.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_1_response = ""  # write at least 8 words
studio_pulse(1, checkpoint_1_response)

---
## 3. Two-Source Interference (2-D)

When two point sources emit waves of the same frequency, they create an **interference pattern** in 2-D space.

At any point $P$, the path difference is:
$$\Delta r = r_2 - r_1$$

| Condition | Path difference | Result |
|-----------|----------------|--------|
| Constructive | $\Delta r = m\lambda$ ($m = 0, \pm 1, \pm 2, \dots$) | Maximum intensity |
| Destructive | $\Delta r = (m + \tfrac{1}{2})\lambda$ | Zero intensity |

The resulting pattern shows bright (constructive) and dark (destructive) **fringes** — similar to what you see when dropping two pebbles in a pond.

---
## 🎬 Interactive Demo 3: Two-Source Interference Pattern (2-D Heatmap)

Visualise the interference pattern from two coherent point sources. Adjust the source **separation** ($d$) and **wavelength** ($\lambda$) to see how the pattern changes.

In [ ]:
def two_source_interference(separation=3.0, wavelength=1.0, t_phase=0.0):
    """Plot 2D interference pattern from two point sources."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Grid
    N = 400
    x = np.linspace(-8, 8, N)
    y = np.linspace(-8, 8, N)
    X, Y = np.meshgrid(x, y)

    # Source positions on the y-axis
    s1 = np.array([0, -separation / 2])
    s2 = np.array([0,  separation / 2])

    k = 2 * np.pi / wavelength
    omega = 2 * np.pi  # normalized frequency

    r1 = np.sqrt((X - s1[0])**2 + (Y - s1[1])**2)
    r2 = np.sqrt((X - s2[0])**2 + (Y - s2[1])**2)

    # Avoid division by zero at sources
    r1 = np.maximum(r1, 0.05)
    r2 = np.maximum(r2, 0.05)

    # Wave fields (with 1/sqrt(r) decay for 2D circular waves)
    psi1 = np.sin(k * r1 - omega * t_phase) / np.sqrt(r1)
    psi2 = np.sin(k * r2 - omega * t_phase) / np.sqrt(r2)
    psi_total = psi1 + psi2

    # Left: instantaneous wave field
    im1 = axes[0].imshow(psi_total, extent=[-8, 8, -8, 8], cmap='RdBu_r',
                          vmin=-2, vmax=2, origin='lower')
    axes[0].plot(*s1, 'ko', ms=8, label='Source 1')
    axes[0].plot(*s2, 'ko', ms=8, label='Source 2')
    axes[0].set_title('Instantaneous Wave Field', fontsize=12)
    axes[0].set_xlabel('x (m)')
    axes[0].set_ylabel('y (m)')
    axes[0].legend()
    plt.colorbar(im1, ax=axes[0], label='Displacement')

    # Right: time-averaged intensity
    # I ~ |A1/sqrt(r1) + A2/sqrt(r2) * e^{i*k*delta_r}|^2
    intensity = 1/r1 + 1/r2 + 2*np.cos(k*(r2 - r1)) / np.sqrt(r1 * r2)
    im2 = axes[1].imshow(intensity, extent=[-8, 8, -8, 8], cmap='hot',
                          origin='lower', vmin=0)
    axes[1].plot(*s1, 'wo', ms=8)
    axes[1].plot(*s2, 'wo', ms=8)
    axes[1].set_title('Time-Averaged Intensity', fontsize=12)
    axes[1].set_xlabel('x (m)')
    axes[1].set_ylabel('y (m)')
    plt.colorbar(im2, ax=axes[1], label='Intensity (arb. units)')

    fig.suptitle(f'd = {separation:.1f} m,   $\\lambda$ = {wavelength:.2f} m,   d/$\\lambda$ = {separation/wavelength:.1f}',
                 fontsize=13, fontweight='bold', y=1.02)
    fig.tight_layout()
    plt.show()

@interact(separation=FloatSlider(min=0.5, max=6.0, step=0.25, value=3.0, description='d (m)'),
          wavelength=FloatSlider(min=0.3, max=3.0, step=0.1, value=1.0, description='$\\lambda$ (m)'),
          t_phase=FloatSlider(min=0, max=2*np.pi, step=0.2, value=0, description='Phase (rad)'))
def show_interference(separation, wavelength, t_phase):
    two_source_interference(separation, wavelength, t_phase)

---
## 4. Standing Waves

When a wave reflects off a boundary, the incident and reflected waves can **superpose** to form a **standing wave**:

$$y(x,t) = 2A \sin(kx) \cos(\omega t)$$

Key features:
- **Nodes**: points that never move ($\sin(kx) = 0$)
- **Antinodes**: points of maximum vibration ($\sin(kx) = \pm 1$)

### Resonant Frequencies on a String (fixed at both ends)

$$f_n = \frac{n}{2L} \sqrt{\frac{T}{\mu}}, \quad n = 1, 2, 3, \dots$$

| Harmonic | $n$ | Wavelength | Name |
|----------|-----|------------|------|
| 1st | 1 | $\lambda_1 = 2L$ | Fundamental |
| 2nd | 2 | $\lambda_2 = L$ | 1st overtone |
| 3rd | 3 | $\lambda_3 = 2L/3$ | 2nd overtone |
| $n$th | $n$ | $\lambda_n = 2L/n$ | $(n-1)$th overtone |

> **Engineering Analogy:** Standing waves on a guitar string produce musical notes. The fundamental frequency determines the pitch, while higher harmonics give the instrument its characteristic tone (timbre). The same principle applies to vibration analysis of bridges and buildings!

---
## 🎬 Interactive Demo 4: Standing Wave Animation

Observe standing waves on a string fixed at both ends. Change the **harmonic number** $n$ to see different modes.

In [ ]:
def standing_wave_animation(n_harmonic=1):
    """Animate a standing wave for the n-th harmonic."""
    L = 1.0  # string length
    A = 1.0
    fig, ax = plt.subplots(figsize=(11, 4))

    x = np.linspace(0, L, 500)
    k = n_harmonic * np.pi / L
    omega = 2 * np.pi * n_harmonic  # proportional to harmonic
    lam = 2 * L / n_harmonic

    # Envelope
    envelope = 2 * A * np.abs(np.sin(k * x))
    ax.fill_between(x, -envelope, envelope, alpha=0.1, color='blue')
    ax.plot(x, envelope, 'b--', lw=1, alpha=0.5)
    ax.plot(x, -envelope, 'b--', lw=1, alpha=0.5)

    line, = ax.plot([], [], 'b-', lw=3)

    # Mark nodes
    nodes_x = np.linspace(0, L, n_harmonic + 1)
    ax.plot(nodes_x, np.zeros_like(nodes_x), 'ko', ms=10, zorder=5, label='Nodes')

    # Mark antinodes
    antinodes_x = (nodes_x[:-1] + nodes_x[1:]) / 2
    ax.plot(antinodes_x, np.zeros_like(antinodes_x), 'r^', ms=10, zorder=5, label='Antinodes')

    ax.set_xlim(-0.02, L + 0.02)
    ax.set_ylim(-2.5, 2.5)
    ax.set_xlabel('Position along string (m)')
    ax.set_ylabel('Displacement (m)')
    ax.set_title(f'Standing Wave: Harmonic n = {n_harmonic},  $\\lambda$ = {lam:.3f} m,  {n_harmonic} half-wavelength(s)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    n_frames = 60

    def animate(i):
        t = i * 2 * np.pi / (omega * n_frames)
        y = 2 * A * np.sin(k * x) * np.cos(omega * t)
        line.set_data(x, y)
        return (line,)

    ani = animation.FuncAnimation(fig, animate, frames=n_frames, interval=40, blit=True)
    plt.close(fig)
    return ani

@interact(n_harmonic=IntSlider(min=1, max=8, step=1, value=1, description='Harmonic n'))
def show_standing(n_harmonic):
    ani = standing_wave_animation(n_harmonic)
    display(HTML(ani.to_jshtml()))

---
## 5. Sound Waves

Sound is a **longitudinal** wave: particles oscillate **parallel** to the direction of propagation, creating alternating regions of **compression** (high pressure) and **rarefaction** (low pressure).

### Speed of Sound

| Medium | Speed (m/s) | Notes |
|--------|-------------|-------|
| Air (20 $^\circ$C) | 343 | $v \approx 331 + 0.6\,T_{\text{C}}$ |
| Water (25 $^\circ$C) | 1480 | Important for sonar |
| Steel | 5960 | Fastest in solids |

### Intensity and Decibels

Sound intensity level in decibels (dB):

$$\beta = 10 \log_{10}\!\left(\frac{I}{I_0}\right)$$

where $I_0 = 10^{-12}$ W/m$^2$ is the threshold of hearing.

| Sound | Intensity (W/m$^2$) | Level (dB) |
|-------|--------------------|-----------|
| Threshold of hearing | $10^{-12}$ | 0 |
| Whisper | $10^{-10}$ | 20 |
| Conversation | $10^{-6}$ | 60 |
| Rock concert | $10^{-1}$ | 110 |
| Threshold of pain | $1$ | 120 |

---
## 🎬 Interactive Demo 5: Sound Wave Visualisation (Pressure Wave)

Visualise a sound wave as a longitudinal pressure variation. The top panel shows the pressure distribution; the bottom shows particle displacement.

In [ ]:
def sound_wave_animation(frequency=2.0, amplitude=1.0):
    """Animate a sound wave showing both pressure and displacement."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

    x = np.linspace(0, 10, 500)
    lam = 343 / (frequency * 100)  # scaled wavelength for visualization
    lam = 2.0  # keep visual scale
    k = 2 * np.pi / lam
    omega = 2 * np.pi * frequency

    # Pressure wave
    line_p, = ax1.plot([], [], 'r-', lw=2.5)
    ax1.set_ylim(-1.8, 1.8)
    ax1.set_ylabel('Pressure variation\n$\\Delta P$ (Pa)', fontsize=11)
    ax1.set_title(f'Sound Wave:  f = {frequency:.1f} Hz (visual scale)', fontsize=12)
    ax1.axhline(0, color='gray', ls='--', lw=0.5)
    ax1.grid(True, alpha=0.3)

    # Add compression/rarefaction labels
    ax1.text(0.5, 1.4, 'C = Compression', color='red', fontsize=9)
    ax1.text(0.5, -1.5, 'R = Rarefaction', color='blue', fontsize=9)

    # Displacement wave (90 deg out of phase with pressure)
    line_d, = ax2.plot([], [], 'b-', lw=2.5)
    ax2.set_ylim(-1.8, 1.8)
    ax2.set_xlabel('Position x (m)')
    ax2.set_ylabel('Particle displacement\n$s$ (m)', fontsize=11)
    ax2.axhline(0, color='gray', ls='--', lw=0.5)
    ax2.grid(True, alpha=0.3)

    # Particle dots for longitudinal visualization
    n_particles = 50
    x_particles = np.linspace(0.2, 9.8, n_particles)
    dots, = ax2.plot([], [], 'ko', ms=4, alpha=0.7)

    fig.tight_layout()

    n_frames = 80
    T_total = 2.0 / frequency if frequency > 0 else 2.0

    def animate(i):
        t = i * T_total / n_frames
        # Pressure: p = P_max * sin(kx - wt)
        p = amplitude * np.sin(k * x - omega * t)
        line_p.set_data(x, p)
        # Displacement: s = s_max * cos(kx - wt) -- 90 deg out of phase
        s = amplitude * np.cos(k * x - omega * t)
        line_d.set_data(x, s)
        # Particle positions (displaced)
        s_part = 0.3 * amplitude * np.cos(k * x_particles - omega * t)
        dots.set_data(x_particles + s_part, np.zeros_like(x_particles))
        return line_p, line_d, dots

    ani = animation.FuncAnimation(fig, animate, frames=n_frames, interval=40, blit=True)
    plt.close(fig)
    return ani

@interact(frequency=FloatSlider(min=0.5, max=4.0, step=0.25, value=2.0, description='f (Hz)'),
          amplitude=FloatSlider(min=0.2, max=1.5, step=0.1, value=1.0, description='A'))
def show_sound(frequency, amplitude):
    ani = sound_wave_animation(frequency, amplitude)
    display(HTML(ani.to_jshtml()))

---
### ⏱️ Checkpoint 2 of 3 — Think · Pair · Explain

Before calculating, predict the direction, sign, or trend of the result. Cite the physical law and one limiting case that supports your prediction.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_2_response = ""  # write at least 8 words
studio_pulse(2, checkpoint_2_response)

---
## 📝 Worked Example 1: Wave on a String

**Problem:** A steel guitar string has length $L = 0.65$ m, mass $m = 3.5$ g, and is under tension $T = 72$ N.

Find: (a) the wave speed, (b) the fundamental frequency, (c) the frequency of the 3rd harmonic.

In [ ]:
# Worked Example 1: Wave on a guitar string
L = 0.65       # m
m = 3.5e-3     # kg
T = 72         # N

# (a) Linear mass density and wave speed
mu = m / L
v = np.sqrt(T / mu)
print(f"Linear mass density: mu = {mu:.4f} kg/m")
print(f"(a) Wave speed: v = sqrt(T/mu) = sqrt({T}/{mu:.4f}) = {v:.1f} m/s")

# (b) Fundamental frequency
f1 = v / (2 * L)
print(f"\n(b) Fundamental frequency: f1 = v/(2L) = {v:.1f}/(2*{L}) = {f1:.1f} Hz")

# (c) 3rd harmonic
f3 = 3 * f1
print(f"\n(c) 3rd harmonic: f3 = 3*f1 = 3*{f1:.1f} = {f3:.1f} Hz")

---
## 📝 Worked Example 2: Two-Source Interference

**Problem:** Two speakers separated by $d = 2.0$ m emit sound at $f = 680$ Hz in phase. A listener stands at a point where $r_1 = 4.0$ m and $r_2 = 4.5$ m from the respective speakers. The speed of sound is 340 m/s.

Is the interference constructive, destructive, or partial?

In [ ]:
# Worked Example 2: Two-source interference
f = 680    # Hz
v_sound = 340  # m/s
r1 = 4.0   # m
r2 = 4.5   # m

# Wavelength
lam = v_sound / f
print(f"Wavelength: lambda = v/f = {v_sound}/{f} = {lam:.3f} m")

# Path difference
delta_r = abs(r2 - r1)
print(f"Path difference: |r2 - r1| = |{r2} - {r1}| = {delta_r:.3f} m")

# How many wavelengths?
ratio = delta_r / lam
print(f"delta_r / lambda = {delta_r:.3f} / {lam:.3f} = {ratio:.2f}")

# Check
if abs(ratio - round(ratio)) < 0.01:
    print(f"\n=> delta_r = {round(ratio)}*lambda -> CONSTRUCTIVE interference (maximum)")
elif abs(ratio - round(ratio) - 0.5) < 0.01 or abs(ratio - round(ratio) + 0.5) < 0.01:
    m_val = int(ratio - 0.5) if ratio > 0 else int(ratio + 0.5)
    print(f"\n=> delta_r = ({m_val} + 1/2)*lambda -> DESTRUCTIVE interference (minimum)")
else:
    print(f"\n=> Partial interference (neither purely constructive nor destructive)")
    print(f"   Closest integer: {round(ratio)}, difference: {abs(ratio - round(ratio)):.2f}")

---
## 🎬 Lab Application: Vibration and Acoustics in Engineering

### Resonance in Structures

Understanding standing waves is critical in engineering:
- **Bridges** can resonate if excited at natural frequencies (cf. Tacoma Narrows)
- **Machine shafts** have critical speeds where vibration amplitudes become dangerous
- **Building floors** are designed to avoid resonance with foot traffic

Below: explore how a bar fixed at both ends vibrates at its natural frequencies.

---
## Problem Set

**Instructions:** Solve each problem analytically first, then verify your answer numerically in the code cell below it. Show your work with clear variable definitions and unit tracking.

- **L1 (Basic):** Straightforward single-concept problems
- **L2 (Intermediate):** Multi-step problems combining two concepts
- **L3 (Challenge):** Multi-concept integration and engineering applications

---
## 🧪 Practice Priority

**L1 problems are the core in-class set.** L2 problems are extensions if time remains; L3 problems are challenges. Nothing here is homework or collected. For every solution use: diagram/configuration → governing law → symbolic setup → units → numerical result → reasonableness check.


### L1 (Basic) — P1

A transverse wave on a string is described by $y(x,t) = 0.05\sin(3.0x - 12t)$ where $x$ and $y$ are in meters and $t$ is in seconds. Find the amplitude, wavelength, frequency, and wave speed.

<details><summary>Answer</summary>$A = 0.05$ m; $\lambda = 2.094$ m; $f = 1.91$ Hz; $v = 4.0$ m/s</details>

In [ ]:
# ✏️ [P1] Your solution here

### L1 (Basic) — P2

A guitar string of length $0.65$ m and mass $3.2$ g is under $73$ N of tension. What is the speed of transverse waves on this string, and what is the fundamental frequency?

<details><summary>Answer</summary>$\mu = 3.2\times10^{-3}/0.65 = 4.9231\times10^{-3}$ kg/m, so $v = \sqrt{73/\mu} = 121.8$ m/s and $f_1 = v/2L = 93.7$ Hz. **[CORRECTED]** previously 122.6 m/s and 94.3 Hz (about 0.7% high).</details>

In [ ]:
# ✏️ [P2] Your solution here

### L1 (Basic) — P3

Two identical speakers emit sound at $f = 850$ Hz in phase. A listener is $5.0$ m from one speaker and $5.5$ m from the other. The speed of sound is $343$ m/s. Determine whether the interference is constructive, destructive, or partial.

<details><summary>Answer</summary>Path difference $= 0.500$ m $= 1.24\lambda$; partial interference (neither perfectly constructive nor destructive)</details>

In [ ]:
# ✏️ [P3] Your solution here

### L1 (Basic) — P4

A sound source produces an intensity of $I = 2.5 \times 10^{-5}$ W/m$^2$. Calculate the sound level in decibels. If the intensity doubles, what is the new decibel level?

<details><summary>Answer</summary>$\beta = 74.0$ dB; doubled: $\beta = 77.0$ dB</details>

In [ ]:
# ✏️ [P4] Your solution here

---
### ⏱️ Checkpoint 3 of 3 — Think · Pair · Explain

Select one L1 solution and explain its diagram, governing law, units, and reasonableness check. Then connect the result to either the Mechatronics or Computer Engineering lens above.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_3_response = ""  # write at least 8 words
studio_pulse(3, checkpoint_3_response)

---
## 🌟 Extension Problems

L2 and L3 problems are optional enrichment, not homework. Use them for remaining studio time or independent curiosity.


### L2 (Intermediate) — P5

A steel wire (density $7800$ kg/m$^3$, diameter $0.80$ mm) is stretched between two supports $1.50$ m apart. The wire vibrates in its third harmonic at $440$ Hz (concert A). Find (a) the wave speed, (b) the tension in the wire, and (c) the frequencies of the first five harmonics.

<details><summary>Answer</summary>(a) $v = 440$ m/s; (b) $T = 760$ N; (c) $f_1 = 146.7$ Hz, $f_2 = 293.3$ Hz, $f_3 = 440$ Hz, $f_4 = 586.7$ Hz, $f_5 = 733.3$ Hz</details>

In [ ]:
# ✏️ [P5] Your solution here

### L2 (Intermediate) — P6

An organ pipe open at both ends has a fundamental frequency of $256$ Hz when the speed of sound is $343$ m/s. (a) What is the length of the pipe? (b) What is the fundamental frequency of a pipe of the same length that is closed at one end? (c) List the first three resonant frequencies for each pipe.

<details><summary>Answer</summary>(a) $L = 0.670$ m; (b) $f_1^\text{closed} = 128$ Hz; (c) Open: $256, 512, 768$ Hz; Closed: $128, 384, 640$ Hz</details>

In [ ]:
# ✏️ [P6] Your solution here

### L2 (Intermediate) — P7

Two coherent sound sources ($f = 1200$ Hz, speed of sound $343$ m/s) are separated by $d = 1.5$ m. Find the angles (measured from the perpendicular bisector) at which the first two maxima and the first minimum occur in the far field.

<details><summary>Answer</summary>$\lambda = 343/1200 = 0.2858$ m, $d = 1.5$ m. Maxima at $d\sin\theta = n\lambda$, minima at $d\sin\theta = (n+\tfrac12)\lambda$: 0th max at $\theta = 0^\circ$; **1st minimum at $\theta = 5.47^\circ$**; **1st maximum at $\theta = 10.98^\circ$**; 2nd maximum at $\theta = 22.40^\circ$. **[CORRECTED]** previously '1st min 11.0$^\circ$, 1st max 22.2$^\circ$' — those are actually the 1st and 2nd *maxima*; the orders were shifted, not merely rounded.</details>

In [ ]:
# ✏️ [P7] Your solution here

### L2 (Intermediate) — P8

A wave pulse $y_1(x,t) = \frac{0.10}{1 + (x - 3t)^2}$ travels to the right, and $y_2(x,t) = \frac{-0.10}{1 + (x + 3t)^2}$ travels to the left (units: m, s). Find (a) the speed of each pulse, (b) the position and time where the pulses completely cancel, and (c) the displacement at $x = 0$ at $t = 0$.

<details><summary>Answer</summary>(a) Both $v = 3.0$ m/s; (b) They cancel at $x = 0$, $t = 0$ (net $y = 0$); (c) $y(0,0) = 0.10 + (-0.10) = 0$</details>

In [ ]:
# ✏️ [P8] Your solution here

### L3 (Challenge) — P9

A vibrating machine is mounted on a concrete floor (speed of longitudinal waves in concrete: $v_c = 3400$ m/s). The machine operates at $60$ Hz. (a) What is the wavelength of vibrations in the floor? (b) Sensitive equipment is located $25$ m away. Assuming the wave intensity decreases as $I \propto 1/r^2$, by how many decibels is the vibration attenuated at that distance compared to $1.0$ m from the machine? (c) If an isolation trench (air gap) of width $0.30$ m is cut into the floor between the machine and the equipment, estimate whether it is effective. Compare the trench width to the wavelength.

<details><summary>Answer</summary>(a) $\lambda = 56.7$ m; (b) $\Delta\beta = 28.0$ dB; (c) Trench width ($0.30$ m) $\ll \lambda$ ($56.7$ m), so the trench is ineffective -- waves diffract around it</details>

In [ ]:
# ✏️ [P9] Your solution here

### L3 (Challenge) — P10

A string of length $L = 2.0$ m, linear density $\mu = 0.010$ kg/m, and tension $T = 40$ N is plucked so that its initial shape is a triangle with peak displacement $h = 0.02$ m at $x = L/3$. This shape can be decomposed into Fourier harmonics. (a) Find the fundamental frequency and wave speed. (b) The Fourier coefficient of the $n$-th harmonic for this triangular pluck is $b_n = \frac{9h}{n^2\pi^2}\sin\!\left(\frac{n\pi}{3}\right)$. Calculate $b_1$, $b_2$, $b_3$. (c) Which harmonics are absent and why?

<details><summary>Answer</summary>(a) $v = \sqrt{T/\mu} = 63.2$ m/s, $f_1 = v/2L = 15.8$ Hz. (b) Using the corrected coefficient $b_n = \dfrac{9h}{n^2\pi^2}\sin\!\left(\dfrac{n\pi}{3}\right)$: $b_1 = 15.794$ mm, $b_2 = 3.949$ mm, $b_3 = 0$. (c) $b_3 = b_6 = b_9 = \cdots = 0$ (all multiples of 3), because the pluck point $x_p = L/3$ is a node of those harmonics. **[CORRECTED]** the coefficient formula originally printed in this problem carried an extra factor of 2 in the denominator, giving values half the correct size. The projection integral $b_n = \frac{2}{L}\int_0^L y(x,0)\sin\frac{n\pi x}{L}dx$ gives the form above; summing it reconstructs the true 20 mm pluck (the halved version reconstructs 10 mm).</details>

In [ ]:
# ✏️ [P10] Your solution here

---
## 🌉 Bridge to Next Week

This week we explored the physics of **mechanical waves and sound**. You learned how waves carry energy through media, how they interfere to produce complex patterns, and how standing waves create resonance.

**Next week (Week 14)** is our **Final Review & Mini Project** session. You will:
- Review all major concepts from the entire semester
- Choose one of three capstone mini-projects to apply your skills
- Demonstrate your final project during the scheduled class

The mini-projects will integrate programming with physics: air resistance projectile analysis, damped oscillation modeling, or rotational energy analysis. Start thinking about which project interests you most!